# Практическая работа № 2. Работа с LLM через API

**Студент:** Архангельский Данила Андреевич  
**Группа:** ИСТ-05  
**Вариант:** 01 (`rental_01.csv`)

Сравнение базового и Few-shot промптов, извлечение дополнительных полей в JSON и оценка по ручной разметке.

In [ ]:
# При необходимости установите зависимости:
# %pip install -U langchain-gigachat pandas pydantic python-dotenv

import os, csv
from pathlib import Path
from typing import Optional
import pandas as pd
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_gigachat.chat_models import GigaChat
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser

load_dotenv()
API_KEY = os.getenv('GIGA_KEY')
if not API_KEY:
    raise RuntimeError('Не найден GIGA_KEY. Добавьте новый ключ в локальный .env рядом с ноутбуком.')
llm = GigaChat(credentials=API_KEY, model='GigaChat-2', verify_ssl_certs=False, temperature=0.2, max_tokens=1200)
print(llm.invoke('Ответь одним словом: работает?').content)

## 1. Загрузка 15 заявок

In [ ]:
VARIANT = '01'
CSV_PATH = Path('data') / f'rental_{VARIANT}.csv'
rows = []
with open(CSV_PATH, encoding='utf-8-sig', newline='') as f:
    reader = csv.reader(f, delimiter=';', quotechar='"')
    header = next(reader, None)
    for row in reader:
        if row:
            rows.append({'amount': int(row[0]), 'text': ';'.join(row[1:]).strip()})
df = pd.DataFrame(rows).head(15).reset_index(drop=True)
assert len(df) == 15
df

## 2. Базовый промпт и оценка

In [ ]:
basic_prompt = PromptTemplate.from_template('''Определи общее количество проживающих в заявке. Учитывай взрослых и детей, не считай животных. Если количество нельзя определить однозначно, верни 0. Верни только целое число.\nЗаявка: {text}\nКоличество:''')
basic_chain = basic_prompt | llm | StrOutputParser()
def run_chain(chain, texts):
    out=[]
    for text in texts:
        try: out.append(chain.invoke({'text': text}))
        except Exception as exc: out.append(f'ERROR: {type(exc).__name__}: {exc}')
    return out
def numeric(series):
    return pd.to_numeric(series.astype(str).str.strip().str.extract(r'^(\\d+)$', expand=False), errors='coerce')
df['result_basic'] = run_chain(basic_chain, df['text'])
basic_num = numeric(df['result_basic'])
acc_basic = basic_num.eq(df['amount']).mean()
print(f'Базовый промпт: {basic_num.eq(df.amount).sum()}/{len(df)} = {acc_basic:.1%}')
df[['amount','result_basic','text']]

## 3. Few-shot и сравнение

In [ ]:
few_prompt = ChatPromptTemplate.from_messages([
 ('system', '''Ты извлекаешь общее число проживающих. Считай взрослых и детей, животных не считай. Примеры: «пара» = 2; «один взрослый и ребёнок» = 2; «2 взрослых и 2 детей» = 4; «автор с женой и двумя детьми» = 4. Если определить невозможно, верни 0. Ответ только целое число.'''),
 ('human', 'Заявка: {text}\\nКоличество проживающих:')
]) 
few_chain = few_prompt | llm | StrOutputParser()
df['result_fewshot'] = run_chain(few_chain, df['text'])
few_num = numeric(df['result_fewshot'])
acc_few = few_num.eq(df['amount']).mean()
print(f'Few-shot: {few_num.eq(df.amount).sum()}/{len(df)} = {acc_few:.1%}')
print(f'Изменение: {acc_few-acc_basic:+.1%}')
df[['amount','result_basic','result_fewshot','text']]

## 4. Извлечение JSON и Pydantic

In [ ]:
class RentalRequest(BaseModel):
    count_adults: Optional[int] = Field(default=None)
    count_children: Optional[int] = Field(default=None)
    start_date: Optional[str] = Field(default=None, description='ДД.ММ')
    nights: Optional[int] = Field(default=None)
    price_per_day: Optional[int] = Field(default=None)
    remarks: Optional[str] = Field(default=None)
parser = JsonOutputParser(pydantic_object=RentalRequest)
prompt = ChatPromptTemplate.from_messages([
 ('system', '''Извлеки поля заявки и верни JSON по схеме. Не выдумывай: отсутствующее значение — null. start_date — самая ранняя дата, формат ДД.ММ. nights — максимальная длительность, если дан диапазон. price_per_day — максимальная цена диапазона. remarks — особые пожелания, животные и удобства.\nПример: 2 взрослых, 1 ребёнок, с 10.08 по 15.08, до 3000 руб/сутки, бассейн => {\\"count_adults\\":2,\\"count_children\\":1,\\"start_date\\":\\"10.08\\",\\"nights\\":5,\\"price_per_day\\":3000,\\"remarks\\":\\"бассейн\\"}'''),
 ('human', 'Заявка: {text}\\nФормат: {format_instructions}')
]).partial(format_instructions=parser.get_format_instructions())
chain = prompt | llm | parser
items=[]
for text in df.text:
    try: items.append(RentalRequest.model_validate(chain.invoke({'text':text})).model_dump())
    except Exception as exc: items.append({'_error':f'{type(exc).__name__}: {exc}'})
ext = pd.DataFrame(items)
for field in RentalRequest.model_fields:
    if field not in ext: ext[field] = None
result = pd.concat([df[['amount','text']],ext],axis=1)
result.to_csv('rental_01_extracted.csv',index=False,encoding='utf-8-sig')
result

## 5. Сравнение с ручной разметкой

In [ ]:
gold = pd.read_csv('gold_fields_01.csv',dtype=str).fillna('')
assert len(gold)==len(df) and gold.text.equals(df.text.astype(str)), 'Gold-разметка не совпадает с набором заявок'
fields=['count_adults','count_children','start_date','nights','price_per_day']
def norm(x):
    if pd.isna(x) or str(x).strip().lower() in {'','nan','none','null'}: return ''
    s=str(x).strip()
    return s[:-2] if s.endswith('.0') else s
scores={}
for field in fields:
    pairs=[(norm(a),norm(b)) for a,b in zip(ext[field],gold[field])]
    scores[field]=sum(a==b for a,b in pairs)/len(pairs)
    print(f'{field}: {sum(a==b for a,b in pairs)}/{len(pairs)} = {scores[field]:.1%}')
print(f'Средняя точность по пяти полям: {sum(scores.values())/len(scores):.1%}')
print('remarks размечено, но не включено в числовую метрику свободного текста.')